# 탄소시계 2단계: 시간 이동 계산 엔진 점검 (Colab용)
**사용법**: 위에서부터 셀을 하나씩 ▶ 실행합니다. (모두 실행도 가능)
1. 셀 1: 구글 드라이브를 연결하고 `project` 폴더를 **자동으로 찾습니다.** (파일이 여러 개면 가장 최근 것을 씀)
2. 셀 2: (필요할 때만) 엔진 파일을 정식 버전으로 되돌립니다.
3. 셀 3: 엔진을 불러오고 **어느 파일을 불러왔는지 확인**합니다.
4. 셀 4: 테스트 14개. **14/14가 나와야** 합니다. 테스트를 고쳐서 통과시키지 마세요. 실패하면 엔진이나 데이터가 사양과 다르다는 뜻입니다.

| 오류 | 원인 | 해결 |
|---|---|---|
| `엔진을 다른 위치에서 불러왔음` | /content에 예전 복사본이 남아 있음 | 셀 3을 다시 실행(자동 삭제) 또는 런타임 → 세션 다시 시작 |
| T1·T2·T6·T10·T12 실패 | 엔진 파일이 예전·수정된 버전 | 셀 2에서 `RESTORE_ENGINE = True`로 바꿔 실행 후 셀 3부터 다시 |
| `ci_dayhour.csv` 후보 0개 | 드라이브에 `data/clean`이 없음 | ZIP을 드라이브에 풀고 01·02 스크립트 실행 |

In [1]:
import os, sys, time
from pathlib import Path
import numpy as np, pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass   # Colab이 아닌 곳에서는 건너뜀

ROOT = Path(os.environ.get("SEARCH_ROOT", "/content/drive"))
hits = sorted(ROOT.rglob("ci_dayhour.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
print(f"ci_dayhour.csv 후보 {len(hits)}개 (최근 수정 순)")
for h in hits:
    print(" -", h, time.strftime("%Y-%m-%d %H:%M", time.localtime(h.stat().st_mtime)))
if not hits:
    raise FileNotFoundError("ci_dayhour.csv를 찾지 못함: ZIP을 드라이브에 풀고 01·02 스크립트를 먼저 실행하세요")

CLEAN = hits[0].parent
PROJECT = CLEAN.parent.parent
SCRIPTS = PROJECT / "scripts"
print("\n사용할 project 폴더:", PROJECT)
for n in ["ci_dayhour.csv", "ci_season_hour.csv", "daily_potential.csv", "activity_presets.csv", "shift_denominators.csv"]:
    print(("OK     " if (CLEAN / n).exists() else "없음!  ") + n)
print(("OK     " if (SCRIPTS / "shift_engine.py").exists() else "없음!  ") + "scripts/shift_engine.py")

Mounted at /content/drive
ci_dayhour.csv 후보 1개 (최근 수정 순)
 - /content/drive/MyDrive/2026 데이터분석연구회/project/data/clean/ci_dayhour.csv 2026-10-27 22:17

사용할 project 폴더: /content/drive/MyDrive/2026 데이터분석연구회/project
OK     ci_dayhour.csv
OK     ci_season_hour.csv
OK     daily_potential.csv
OK     activity_presets.csv
OK     shift_denominators.csv
OK     scripts/shift_engine.py


### 셀 2: (필요할 때만) 엔진을 정식 버전으로 되돌리기
엔진 파일이 예전 버전이거나 다른 도구로 수정되어 테스트가 맞지 않을 때만 `RESTORE_ENGINE = True`로 바꿔 실행하세요. 평소에는 `False`로 두면 아무 일도 하지 않습니다.

In [2]:
ENGINE_SOURCE = r'''
"""
[2단계] 시간 이동 계산 엔진 (앱의 계산 담당)
- 1단계 결과 CSV(ci_dayhour.csv 등)를 읽어서 '이 전기를 다른 시간으로 옮기면 몇 g이 줄어드는가'를 계산한다.
- 앱(3단계)은 이 파일을 불러와서(import) 쓰기만 한다. 계산 규칙을 한곳에 모아 두는 이유는
  (1) 숫자가 화면마다 달라지는 것을 막고 (2) 테스트로 계산이 맞는지 확인하기 위해서이다.

세 부분으로 읽으면 된다.
  Part 1. 시간 규칙 : 시계 시각 <-> 거래시간 변환
  Part 2. 탄소집약도 조회 : 계절·평일/휴일·월·계수 세트로 시간별 값을 구함
  Part 3. 이동 계산 : 활동 1건의 감소량(g), 감소율 3가지 분모, 가장 깨끗한 시작 시각
"""
from pathlib import Path
import numpy as np
import pandas as pd

SCENARIOS = {"mid": "ci_mid", "min": "ci_min", "max": "ci_max", "direct": "ci_direct"}
SCENARIO_LABEL = {"mid": "IPCC 전과정 중앙값(기본)", "min": "IPCC 전과정 최소", "max": "IPCC 전과정 최대", "direct": "직접배출"}
EVENING = (19, 2)   # 저녁 19~21시: (시작 시각, 길이)
NOON = (12, 2)      # 정오 12~14시


# =============================================================== Part 1. 시간 규칙
def clock_to_trading(clock_hour: int) -> int:
    """시계 시각(0~23, 그 시각부터 1시간) -> 거래시간(1~24).
    거래시간 h는 (h-1)시~h시 구간이므로, 19시~20시는 거래시간 20이다."""
    if not (0 <= int(clock_hour) <= 23):
        raise ValueError(f"시각은 0~23이어야 함: {clock_hour}")
    return int(clock_hour) + 1


def trading_hours(start_clock: int, duration: int) -> list:
    """start_clock 시작, duration 시간 동안의 거래시간 목록 (자정을 넘으면 0시로 이어짐)."""
    if not (0 <= int(start_clock) <= 23):
        raise ValueError(f"시작 시각은 0~23이어야 함: {start_clock}")
    if not (1 <= int(duration) <= 24):
        raise ValueError(f"길이는 1~24시간이어야 함: {duration}")
    return [clock_to_trading((int(start_clock) + k) % 24) for k in range(int(duration))]


class ShiftEngine:
    def __init__(self, clean_dir):
        clean_dir = Path(clean_dir)
        self.dh = pd.read_csv(clean_dir / "ci_dayhour.csv")
        self.daily = pd.read_csv(clean_dir / "daily_potential.csv")
        self.presets = pd.read_csv(clean_dir / "activity_presets.csv")
        self._cache = {}

    # =========================================================== Part 2. 탄소집약도 조회
    def _table(self, season=None, day_type=None, month=None):
        """거래시간별 (거래량 합, 배출량 합, 탄소집약도) 표. 같은 조건은 한 번만 계산한다."""
        key = (season, day_type, month)
        if key in self._cache:
            return self._cache[key]
        d = self.dh
        if season is not None:   d = d[d.season == season]
        if day_type is not None: d = d[d.day_type == day_type]
        if month is not None:    d = d[d.month == month]
        if d.empty:
            raise ValueError(f"조건에 맞는 자료가 없음: {key}")
        out = pd.DataFrame({"mwh": d.groupby("hour")["total_mwh"].sum()})
        for sc, col in SCENARIOS.items():
            out["em_" + sc] = (d[col] * d["total_mwh"]).groupby(d["hour"]).sum()   # 배출량 = 집약도 x 거래량
            out["ci_" + sc] = out["em_" + sc] / out["mwh"]                         # 가중평균 탄소집약도
        self._cache[key] = out
        return out

    def hourly_ci(self, season=None, day_type=None, month=None, scenario="mid") -> pd.Series:
        """거래시간 1~24별 탄소집약도(gCO2eq/kWh). 조건을 None으로 두면 전체."""
        return self._table(season, day_type, month)["ci_" + scenario]

    def window_ci(self, start_clock, duration=1, season=None, day_type=None, month=None, scenario="mid") -> float:
        """활동이 start_clock부터 duration시간 일정한 전력으로 돌 때의 평균 탄소집약도."""
        ci = self.hourly_ci(season, day_type, month, scenario)
        return float(ci.loc[trading_hours(start_clock, duration)].mean())

    def ci_at(self, date, start_clock, scenario="mid") -> float:
        """특정 날짜의 특정 시각 탄소집약도 (손계산 대조용)."""
        row = self.dh[(self.dh.date == str(date)) & (self.dh.hour == clock_to_trading(start_clock))]
        if row.empty:
            raise ValueError(f"해당 날짜·시각 자료 없음: {date} {start_clock}시")
        return float(row[SCENARIOS[scenario]].iloc[0])

    # =========================================================== Part 3. 이동 계산
    def shift(self, kwh, from_start, to_start, duration=1, season=None, day_type=None, month=None, scenario="mid") -> dict:
        """활동 1건(kwh)을 from_start 시각에서 to_start 시각으로 옮길 때의 결과.
        전력량은 그대로(옮기기만 하고 줄이지 않음)이므로 kwh_before == kwh_after 이다."""
        if kwh < 0:
            raise ValueError("kWh는 0 이상이어야 함")
        ci0 = self.window_ci(from_start, duration, season, day_type, month, scenario)
        ci1 = self.window_ci(to_start, duration, season, day_type, month, scenario)
        before, after = kwh * ci0, kwh * ci1
        saved = before - after
        rate = saved / before if before > 0 else float("nan")
        return dict(kwh_before=kwh, kwh_after=kwh, ci_from=ci0, ci_to=ci1, g_before=before, g_after=after,
                    saved_g=saved, saved_rate=rate,       # 이동분 기준 감소율. '안 쓰고 줄이기' 대비 옮기기의 효과와 같은 값
                    reduce_g=before)                      # 같은 전력을 아예 쓰지 않을 때의 감소량(비교용)

    def shift_band(self, kwh, from_start, to_start, duration=1, season=None, day_type=None, month=None) -> pd.DataFrame:
        """계수 4세트로 같은 이동을 계산해 불확실성 범위를 보여 줌."""
        rows = []
        for sc in SCENARIOS:
            r = self.shift(kwh, from_start, to_start, duration, season, day_type, month, sc)
            rows.append(dict(scenario=SCENARIO_LABEL[sc], ci_from=r["ci_from"], ci_to=r["ci_to"], saved_g=r["saved_g"], saved_rate=r["saved_rate"]))
        return pd.DataFrame(rows)

    def denominators(self, p, season=None, day_type=None, month=None, scenario="mid") -> dict:
        """저녁(19~21시) 사용의 p를 정오(12~14시)로 옮길 때 감소율 3가지 분모 (교수님 피드백).
        (가) 이동한 전력의 원래 배출 (나) 저녁 19~21시 총 배출 (다) 하루 전체 배출"""
        t = self._table(season, day_type, month)
        eh, nh = trading_hours(*EVENING), trading_hours(*NOON)
        eve = t.loc[eh, "em_" + scenario].sum() / t.loc[eh, "mwh"].sum()       # 저녁 가중평균 CI
        noon = t.loc[nh, "em_" + scenario].sum() / t.loc[nh, "mwh"].sum()
        gap = eve - noon
        day_avg = t["em_" + scenario].sum() / t["mwh"].sum()                    # 하루 가중평균 CI
        share_eve = t.loc[eh, "mwh"].sum() / t["mwh"].sum()                     # 저녁이 하루 사용에서 차지하는 비중(시장 거래량 모양)
        uniform = p * (len(eh) / 24) * gap / t["ci_" + scenario].mean()         # 24시간 고르게 쓴다고 가정
        return dict(p=p, noon_ci=noon, eve_ci=eve, gap_g_per_kwh=gap,
                    moved_rate=gap / eve,                                       # (가)
                    evening_rate=p * gap / eve,                                 # (나)
                    daily_rate_market=p * share_eve * gap / day_avg,            # (다) 시장 거래량 모양
                    daily_rate_uniform=uniform,                                 # (다) 고르게 사용
                    evening_share_of_day=share_eve)

    def best_start(self, duration=1, earliest=7, latest_end=22, reference_start=19, season=None, day_type=None,
                   month=None, scenario="mid") -> pd.DataFrame:
        """허용 시간 안(earliest시 이후 시작, latest_end시 이전 종료)에서 가장 깨끗한 시작 시각 순위."""
        rows = []
        ref = self.window_ci(reference_start, duration, season, day_type, month, scenario)
        for s in range(earliest, latest_end - duration + 1):
            c = self.window_ci(s, duration, season, day_type, month, scenario)
            rows.append(dict(start=s, end=s + duration, ci=c, saved_rate_vs_ref=(ref - c) / ref))
        return pd.DataFrame(rows).sort_values("ci").reset_index(drop=True)

    def daily_distribution(self, season=None, day_type=None) -> pd.Series:
        """날짜별 이동분 감소율(저녁->정오, %) 분포. 앱에서 히스토그램으로 보여 줌."""
        d = self.daily
        if season is not None:   d = d[d.season == season]
        if day_type is not None: d = d[d.day_type == day_type]
        return d["red_to_noon_pct"]

    def activity_defaults(self, category, ef_g_per_kwh=None) -> dict:
        """에너지공단 품목의 라벨 CO2(g) 대표값. 환산계수가 있으면 kWh도 계산하고, 없으면 kWh는 None."""
        r = self.presets[self.presets.category == category]
        if r.empty:
            raise ValueError(f"품목 없음: {category}")
        r = r.iloc[0]
        out = dict(category=category, use_unit=str(r.use_unit), co2_label_median=float(r.co2_median), co2_label_p25=float(r.co2_p25), co2_label_p75=float(r.co2_p75))
        out.update({"kwh_median": None, "kwh_p25": None, "kwh_p75": None})
        if ef_g_per_kwh:
            out.update(kwh_median=r.co2_median / ef_g_per_kwh, kwh_p25=r.co2_p25 / ef_g_per_kwh, kwh_p75=r.co2_p75 / ef_g_per_kwh)
        return out
'''

RESTORE_ENGINE = False   # True로 바꾸고 실행하면 scripts/shift_engine.py를 정식 버전으로 덮어씀
if RESTORE_ENGINE:
    (SCRIPTS / 'shift_engine.py').write_text(ENGINE_SOURCE, encoding='utf-8')
    print('복원 완료:', SCRIPTS / 'shift_engine.py')
else:
    print('복원하지 않음 (RESTORE_ENGINE = False)')

복원하지 않음 (RESTORE_ENGINE = False)


### 셀 3: 엔진 불러오기와 확인
- `/content`에 남아 있을 수 있는 예전 복사본을 지우고, 이미 불러온 모듈도 비웁니다.
- 프로젝트 `scripts` 폴더를 검색 경로 **맨 앞**에 넣습니다(`append`가 아니라 `insert(0)`).
- 마지막에 **어느 파일을 불러왔는지** 확인합니다. 이 확인이 없으면 엔진이 두 개일 때 눈치채기 어렵습니다.

In [3]:
stale = Path("/content/shift_engine.py")
if stale.exists():
    stale.unlink(); print("삭제한 예전 복사본:", stale)
sys.modules.pop("shift_engine", None)
sys.path[:] = [p for p in sys.path if p != str(SCRIPTS)]
sys.path.insert(0, str(SCRIPTS))

import shift_engine
from shift_engine import ShiftEngine, trading_hours, clock_to_trading
print("불러온 엔진 파일:", shift_engine.__file__)
assert Path(shift_engine.__file__).resolve() == (SCRIPTS / "shift_engine.py").resolve(), "엔진을 다른 위치에서 불러왔음!"
print("확인 완료: 프로젝트 폴더의 엔진을 사용 중")

불러온 엔진 파일: /content/drive/MyDrive/2026 데이터분석연구회/project/scripts/shift_engine.py
확인 완료: 프로젝트 폴더의 엔진을 사용 중


### 셀 4: 테스트 14개
**테스트는 '정답 기준(사양)'입니다.** 엔진이 틀렸으면 엔진을 고치고, 테스트를 엔진에 맞춰 고치지 않습니다. (예: 시각 변환 T1의 `[24, 1]`, 시간별 값 T2의 `387.0 / 429.5`는 1단계에서 확인한 규칙입니다.)

In [4]:
E = ShiftEngine(CLEAN)
results = []
def check(name, cond, detail=""):
    results.append(bool(cond)); print(("PASS " if cond else "FAIL ") + name + (f"  [{detail}]" if detail else ""))

# T1 거래시간 변환: 19시~20시 = 거래시간 20, 자정을 넘으면 이어짐
check("T1 시각 변환", clock_to_trading(19) == 20 and trading_hours(23, 2) == [24, 1], f"{trading_hours(23,2)}")
# T2 1단계 결과와 같은가: 연평균 시간별 값(최저 13=387.0, 최고 21=429.5)
h = E.hourly_ci()
check("T2 시간별 최저·최고", h.idxmin() == 13 and h.idxmax() == 21 and abs(h.loc[13]-387.0) < .1 and abs(h.loc[21]-429.5) < .1, f"13시:{h.loc[13]:.1f} 21시:{h.loc[21]:.1f}")
# T3 정오 388.9, 저녁 429.3, 이동분 기준 감소율 9.4%
d = E.denominators(0.1)
check("T3 정오·저녁·감소율", abs(d["noon_ci"]-388.9) < .1 and abs(d["eve_ci"]-429.3) < .1 and abs(d["moved_rate"]-0.0941) < .0005,
      f"정오 {d['noon_ci']:.1f} 저녁 {d['eve_ci']:.1f} (가){d['moved_rate']*100:.2f}%")
# T4 분모 3가지가 02 스크립트 결과(shift_denominators.csv)와 같은가 (연간 + 봄 평일)
sd = pd.read_csv(CLEAN / "shift_denominators.csv"); ok = True
for scope, kw in [("연간", {}), ("봄 평일", dict(season="봄", day_type="평일"))]:
    for p in (10, 20, 30):
        row = sd[(sd.scope == scope) & (sd.p_pct == p)].iloc[0]; x = E.denominators(p/100, **kw)
        ok &= abs(x["moved_rate"]*100-row.moved_pct) < .01 and abs(x["evening_rate"]*100-row.evening_pct) < .01
        ok &= abs(x["daily_rate_market"]*100-row.daily_actual_pct) < .01 and abs(x["daily_rate_uniform"]*100-row.daily_uniform_pct) < .01
check("T4 분모 3가지 = 02 스크립트 결과", ok)
# T5 전력량 보존: 옮기기만 하므로 전후 kWh가 같다
r = E.shift(10, 19, 12, 2)
check("T5 전력량 보존", r["kwh_before"] == r["kwh_after"] == 10)
# T6 되돌리기 대칭: 옮겼다가 되돌리면 감소량의 부호만 바뀐다
a, b = E.shift(5, 19, 12, 2), E.shift(5, 12, 19, 2)
check("T6 대칭", abs(a["saved_g"] + b["saved_g"]) < 1e-9, f"{a['saved_g']:.2f} / {b['saved_g']:.2f}")
# T7 같은 시각으로 옮기면 감소 0
check("T7 같은 시각 = 0", abs(E.shift(3, 14, 14, 1)["saved_g"]) < 1e-9)
# T8 계수 세트 순서: 최소 <= 중앙 <= 최대 (24시간 모두)
hm = [E.hourly_ci(scenario=s) for s in ("min", "mid", "max")]
check("T8 최소<=중앙<=최대", bool(((hm[0] <= hm[1]) & (hm[1] <= hm[2])).all()))
# T9 손계산 대조: 2025-04-15 평일 13~14시(거래시간 14)의 탄소집약도 = 308.372 (손계산 308.4)
check("T9 손계산 대조", abs(E.ci_at("2025-04-15", 13) - 308.372) < .01, f"{E.ci_at('2025-04-15', 13):.3f}")
# T10 잘못된 입력은 오류를 낸다
errs = 0
for f in (lambda: E.shift(-1, 19, 12), lambda: E.window_ci(24, 1), lambda: E.window_ci(10, 0), lambda: E.hourly_ci(season="봄봄")):
    try: f()
    except ValueError: errs += 1
check("T10 잘못된 입력 오류", errs == 4, f"{errs}/4")
# T11 날짜별 분포: 365일, 중앙값 약 9.5%, 상위 10% 약 21.5%
dd = E.daily_distribution()
check("T11 날짜별 분포", len(dd) == 365 and abs(dd.median()-9.53) < .05 and abs(dd.quantile(.9)-21.5) < .1, f"중앙 {dd.median():.2f} 상위10% {dd.quantile(.9):.2f}")
# T12 가장 깨끗한 시작 시각 순위가 허용 범위 안이고, 기준 시각(19시)보다 나쁘지 않다
bs = E.best_start(duration=2, season="봄", day_type="평일")
check("T12 가장 깨끗한 시각", bs.start.min() >= 7 and bs.end.max() <= 22 and bs.saved_rate_vs_ref.iloc[0] >= 0, f"1위 {int(bs.start.iloc[0])}~{int(bs.end.iloc[0])}시")

# T13 엔진 안의 두 계산이 서로 맞는가: shift()의 1kWh 감소량 = denominators()의 격차(g/kWh)
gap = E.denominators(0.1)["gap_g_per_kwh"]; one = E.shift(1, 19, 12, 2)["saved_g"]
check("T13 shift = denominators 일관성", abs(one - gap) < 0.5, f"shift {one:.2f} g / 격차 {gap:.2f} g")
# T14 시간별 값의 가중평균이 연평균(414.894)과 같은가: 시간별 조회가 거래량 가중평균이라는 증거
t = E._table(); wavg = (t["ci_mid"] * t["mwh"]).sum() / t["mwh"].sum()
check("T14 시간별 가중평균 = 연평균", abs(wavg - 414.894) < 0.05, f"{wavg:.3f}")

print(f"\n결과: {sum(results)}/{len(results)} 통과")

PASS T1 시각 변환  [[24, 1]]
PASS T2 시간별 최저·최고  [13시:387.0 21시:429.5]
PASS T3 정오·저녁·감소율  [정오 388.9 저녁 429.3 (가)9.41%]
PASS T4 분모 3가지 = 02 스크립트 결과
PASS T5 전력량 보존
PASS T6 대칭  [202.07 / -202.07]
PASS T7 같은 시각 = 0
PASS T8 최소<=중앙<=최대
PASS T9 손계산 대조  [308.372]
PASS T10 잘못된 입력 오류  [4/4]
PASS T11 날짜별 분포  [중앙 9.53 상위10% 21.48]
PASS T12 가장 깨끗한 시각  [1위 12~14시]
PASS T13 shift = denominators 일관성  [shift 40.41 g / 격차 40.39 g]
PASS T14 시간별 가중평균 = 연평균  [414.894]

결과: 14/14 통과


### 셀 5: 사용 예시 (계수 세트별, 계절·요일별, 가장 깨끗한 시각)

In [5]:
# ------------------------------------------------------------------ 사용 예시
print("\n[예시 1] 1kWh를 19시→12시로 2시간 활동 이동 (계수 세트별)")
print(E.shift_band(1, 19, 12, 2).round(3).to_string(index=False))
print("\n[예시 2] 계절x평일/휴일: 1kWh, 19시→12시, 2시간")
rows = []
for s in ["봄", "여름", "가을", "겨울"]:
    for dt in ["평일", "휴일"]:
        r = E.shift(1, 19, 12, 2, s, dt); rows.append(dict(계절=s, 구분=dt, 감소g=r["saved_g"], 감소율=r["saved_rate"]))
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("\n[예시 3] 봄 평일, 2시간 활동 가장 깨끗한 시작 시각 상위 3 (07~22시 안)")
print(E.best_start(2, season="봄", day_type="평일").head(3).round(3).to_string(index=False))
print("\n[예시 4] 라벨 CO2 대표값 (환산계수 미정이라 kWh는 None)")
for c in ["전기세탁기(일반)", "의류건조기"]:
    print(E.activity_defaults(c))


[예시 1] 1kWh를 19시→12시로 2시간 활동 이동 (계수 세트별)
        scenario  ci_from   ci_to  saved_g  saved_rate
IPCC 전과정 중앙값(기본)  429.338 388.923   40.415       0.094
     IPCC 전과정 최소  372.062 335.846   36.216       0.097
     IPCC 전과정 최대  564.223 517.509   46.714       0.083
            직접배출  360.142 324.729   35.412       0.098

[예시 2] 계절x평일/휴일: 1kWh, 19시→12시, 2시간
계절 구분    감소g   감소율
 봄 평일 54.711 0.141
 봄 휴일 47.650 0.136
여름 평일 34.859 0.073
여름 휴일 49.978 0.111
가을 평일 35.404 0.077
가을 휴일 47.084 0.111
겨울 평일 32.419 0.076
겨울 휴일 46.045 0.118

[예시 3] 봄 평일, 2시간 활동 가장 깨끗한 시작 시각 상위 3 (07~22시 안)
 start  end      ci  saved_rate_vs_ref
    12   14 332.103              0.141
    11   13 332.431              0.141
    13   15 337.396              0.128

[예시 4] 라벨 CO2 대표값 (환산계수 미정이라 kWh는 None)
{'category': '전기세탁기(일반)', 'use_unit': '1회', 'co2_label_median': 40.0, 'co2_label_p25': 33.0, 'co2_label_p75': 48.0, 'kwh_median': None, 'kwh_p25': None, 'kwh_p75': None}
{'category': '의류건조기', 'use_unit': '1회', 'co2_label_median'

### 셀 6: 내 값으로 해 보기
아래 값을 바꿔서 실행해 보세요. 시각은 **시계 시각**(0~23, 그 시각부터 1시간)입니다. 결과는 팀원이 손으로 한 번 검산해 보세요.

In [6]:
kwh, from_start, to_start, duration = 1.0, 19, 12, 2     # 1kWh, 19시 시작 -> 12시 시작, 2시간 활동
season, day_type = "봄", "평일"                          # None으로 두면 전체

r = E.shift(kwh, from_start, to_start, duration, season, day_type)
print(f"{from_start}시 -> {to_start}시 ({duration}시간), {season} {day_type}, {kwh}kWh")
print(f"  탄소집약도: {r['ci_from']:.1f} -> {r['ci_to']:.1f} g/kWh")
print(f"  배출량   : {r['g_before']:.1f} g -> {r['g_after']:.1f} g (감소 {r['saved_g']:.1f} g, 이동분 기준 {r['saved_rate']*100:.1f}%)")
print(f"  (참고) 같은 전력을 아예 쓰지 않으면 {r['reduce_g']:.1f} g 감소")
print(E.best_start(duration, season=season, day_type=day_type).head(3).round(3).to_string(index=False))

19시 -> 12시 (2시간), 봄 평일, 1.0kWh
  탄소집약도: 386.8 -> 332.1 g/kWh
  배출량   : 386.8 g -> 332.1 g (감소 54.7 g, 이동분 기준 14.1%)
  (참고) 같은 전력을 아예 쓰지 않으면 386.8 g 감소
 start  end      ci  saved_rate_vs_ref
    12   14 332.103              0.141
    11   13 332.431              0.141
    13   15 337.396              0.128
